# SafeShift CT1 — Preflight CPU cục bộ
Chỉ kiểm C1, metadata gói Python và ghi/đọc/xóa scratch 64 KiB. Không cài dependency, đọc dataset/weights, gọi GPU, Kaggle hoặc tiêu lượt model. Ghi một receipt mới tại SAFESHIFT_CT1_OUTPUT_ROOT.


In [ ]:
from pathlib import Path
import os
import sys

repo = Path(os.environ.get("SAFESHIFT_CT1_REPO", ".")).resolve(strict=True)
assert (repo / "scripts/ct1_pilot.py").is_file(), "Set SAFESHIFT_CT1_REPO to the reviewed checkout"
assert sys.version_info >= (3, 11), "Python 3.11+ required for CPU preparation"
if os.name == "nt" and repo.drive.upper() != "D:":
    raise ValueError("SafeShift workspace must be on D:")
workspace = Path(os.environ.get("SAFESHIFT_CT1_OUTPUT_ROOT", repo / "data/processed")).resolve()
if os.name == "nt" and workspace.drive.upper() != "D:":
    raise ValueError("SafeShift output must be on D:")
workspace.mkdir(parents=True, exist_ok=True)
offline_env = {**os.environ, "TMPDIR": str(workspace), "TEMP": str(workspace),
               "TMP": str(workspace), "PYTHONDONTWRITEBYTECODE": "1", "HF_HUB_OFFLINE": "1",
               "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": ""}


In [ ]:
import hashlib
from importlib import metadata
import json
import platform
import shutil
import tempfile

c1 = (repo / "configs/pilot/classification_c1.txt").read_bytes()
assert hashlib.sha256(c1).hexdigest() == "816a9c2602fc6ab8c4589a0df22b9d7063f9427b7bda2554788f8c92a95bfbe3"
with tempfile.TemporaryDirectory(prefix="ct1-probe-", dir=workspace) as scratch:
    test_file = Path(scratch) / "bounded-write.bin"
    block = bytes(64 * 1024)
    with test_file.open("xb") as stream:
        stream.write(block)
        stream.flush()
        os.fsync(stream.fileno())
    assert test_file.read_bytes() == block

packages = {}
for package in ("torch", "transformers", "accelerate", "Pillow"):
    try:
        packages[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        packages[package] = None

receipt = {
    "status": "CPU_ENVIRONMENT_OBSERVED", "release_status": "BLOCKED",
    "platform": platform.system(), "machine": platform.machine(),
    "python": platform.python_version(), "abi": sys.implementation.cache_tag,
    "package_metadata": packages, "scratch_free_bytes": shutil.disk_usage(workspace).free,
    "bounded_fsync_probe_bytes": len(block), "input_check": "NOT_RUN",
    "weights": "NOT_OPENED", "gpu": "NOT_RUN", "native_calls": 0,
}
with (workspace / "ct1-cpu-preflight.json").open("x", encoding="utf-8") as stream:
    json.dump(receipt, stream, indent=2, allow_nan=False)
    stream.write("\n")
print(receipt)


## Điều kiện live còn thiếu
Xác minh Input marker duy nhất, source/model/processor/weights hash và quyền. Kiểm cgroup/RAM, disk/quota, GPU/driver/CUDA/FP16 child và map 2×T4 trong môi trường được cấp quyền. Đo tải weights, synthetic/approved first-call và export trước pilot. Xem docs/KAGGLE_FAILURES_AND_RELEASE_GATES.md.
